# 05 Models — XGBoost — Women's

XGBoost is the consensus top performer for this competition (research). We use shallow trees with regularization and LOGO-CV for honest evaluation.

**Approach:**
- LOGO cross-validation (each season = one fold)
- Train on all other seasons, predict held-out season's tournament games
- Evaluate with Brier score (MSE of predicted probabilities vs actual outcomes)
- Train final model on all data for Stage 1/2 predictions
- Apply Platt scaling (isotonic regression) for calibration

**Inputs** (from S3 `04_preprocessing/womens/`):
- `train_features.parquet`, `stage1_features.parquet`, `stage2_features.parquet`, `feature_columns.parquet`

**Outputs** (to S3 `05_models/xgboost/womens/`):
- `oof_predictions.parquet` — out-of-fold predictions for all training matchups
- `stage1_predictions.parquet` — predictions for Stage 1 grid
- `stage2_predictions.parquet` — predictions for Stage 2 grid
- `cv_results.parquet` — per-fold Brier scores

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except:
    !pip install 'xgboost>=2.0,<3.0'
    import xgboost as xgb

from sklearn.metrics import brier_score_loss
from sklearn.isotonic import IsotonicRegression

print(f"XGBoost version: {xgb.__version__}")

XGBoost version: 2.1.4


#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 or local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"../../04_preprocessing/output/{filename}")

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "womens"
STAGE = "05_models/xgboost"
INPUT_PREFIX = f"s3://{BUCKET}/04_preprocessing/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
train = read_parquet("train_features.parquet")
stage1 = read_parquet("stage1_features.parquet")
stage2 = read_parquet("stage2_features.parquet")
feature_cols = read_parquet("feature_columns.parquet")['feature'].tolist()

print(f"Training data: {train.shape}")
print(f"Stage 1 grid: {stage1.shape}")
print(f"Stage 2 grid: {stage2.shape}")
print(f"Features: {len(feature_cols)}")

Training data: (3434, 64)
Stage 1 grid: (258131, 62)
Stage 2 grid: (65703, 61)
Features: 21


## 2. XGBoost Hyperparameters

Research recommends shallow trees (max depth 2–4) with regularization. We use conservative settings to avoid overfitting on the limited tournament game dataset.

In [6]:
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'max_depth': 3,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'seed': 42,
    'verbosity': 0,
}

N_ROUNDS = 500
EARLY_STOPPING = 50

print("XGBoost parameters:")
for k, v in xgb_params.items():
    print(f"  {k}: {v}")
print(f"  n_rounds: {N_ROUNDS}")
print(f"  early_stopping: {EARLY_STOPPING}")

XGBoost parameters:
  objective: binary:logistic
  eval_metric: logloss
  max_depth: 3
  learning_rate: 0.05
  subsample: 0.8
  colsample_bytree: 0.8
  min_child_weight: 5
  reg_alpha: 1.0
  reg_lambda: 1.0
  seed: 42
  verbosity: 0
  n_rounds: 500
  early_stopping: 50


## 3. LOGO Cross-Validation

Train on all folds except one, predict the held-out fold. Collect out-of-fold predictions for all training matchups.

In [7]:
# Only use the original (non-flipped) rows for OOF evaluation
# The flipped rows are for training only
# Original rows have TeamA < TeamB (submission format)
train_original = train[train['TeamA'] < train['TeamB']].copy()

folds = sorted(train['Fold'].unique())
print(f"Total folds: {len(folds)}")
print(f"Original (non-flipped) matchups: {len(train_original)}")

Total folds: 27
Original (non-flipped) matchups: 1717


In [8]:
oof_preds = []
cv_results = []

for fold in folds:
    # Train on all folds except current (use flip-doubled data for training)
    train_mask = train['Fold'] != fold
    val_mask = (train_original['Fold'] == fold)
    
    X_train = train.loc[train_mask, feature_cols]
    y_train = train.loc[train_mask, 'Label']
    X_val = train_original.loc[val_mask, feature_cols]
    y_val = train_original.loc[val_mask, 'Label']
    
    if len(X_val) == 0:
        continue
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=N_ROUNDS,
        evals=[(dval, 'val')],
        early_stopping_rounds=EARLY_STOPPING,
        verbose_eval=False
    )
    
    # Explicitly use best iteration for predict (required in xgboost < 2.0)
    preds = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
    brier = brier_score_loss(y_val, preds)
    
    cv_results.append({
        'Fold': fold,
        'BrierScore': brier,
        'Games': len(y_val),
        'BestRound': model.best_iteration
    })
    
    fold_oof = train_original.loc[val_mask, ['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val']].copy()
    fold_oof['Pred'] = preds
    oof_preds.append(fold_oof)
    
    print(f"  Fold {fold}: Brier={brier:.4f}, Games={len(y_val)}, BestRound={model.best_iteration}")

oof_df = pd.concat(oof_preds, ignore_index=True)
cv_df = pd.DataFrame(cv_results)

  Fold 1998: Brier=0.1745, Games=63, BestRound=49
  Fold 1999: Brier=0.1375, Games=63, BestRound=129
  Fold 2000: Brier=0.1408, Games=63, BestRound=435
  Fold 2001: Brier=0.1492, Games=63, BestRound=73
  Fold 2002: Brier=0.1327, Games=63, BestRound=218
  Fold 2003: Brier=0.1179, Games=63, BestRound=192
  Fold 2004: Brier=0.1579, Games=63, BestRound=110
  Fold 2005: Brier=0.1588, Games=63, BestRound=113
  Fold 2006: Brier=0.1314, Games=63, BestRound=77
  Fold 2007: Brier=0.1737, Games=63, BestRound=59
  Fold 2008: Brier=0.1226, Games=63, BestRound=242
  Fold 2009: Brier=0.1553, Games=63, BestRound=76
  Fold 2010: Brier=0.1419, Games=63, BestRound=99
  Fold 2011: Brier=0.1354, Games=63, BestRound=124
  Fold 2012: Brier=0.1186, Games=63, BestRound=183
  Fold 2013: Brier=0.1498, Games=63, BestRound=105
  Fold 2014: Brier=0.1330, Games=63, BestRound=128
  Fold 2015: Brier=0.1222, Games=63, BestRound=126
  Fold 2016: Brier=0.1655, Games=63, BestRound=106
  Fold 2017: Brier=0.1406, Games=63, 

In [9]:
# Overall CV Brier score
overall_brier = brier_score_loss(oof_df['Label'], oof_df['Pred'])
stage1_oof = oof_df[oof_df['IsStage1Val'] == 1]
stage1_brier = brier_score_loss(stage1_oof['Label'], stage1_oof['Pred']) if len(stage1_oof) > 0 else None

print(f"\nOverall OOF Brier Score: {overall_brier:.4f}")
print(f"Stage 1 (2022-2025) Brier Score: {stage1_brier:.4f}" if stage1_brier else "No Stage 1 data")
print(f"\nCV Results Summary:")
print(f"  Mean Brier: {cv_df['BrierScore'].mean():.4f}")
print(f"  Std Brier: {cv_df['BrierScore'].std():.4f}")
print(f"  Min Brier: {cv_df['BrierScore'].min():.4f} (Fold {cv_df.loc[cv_df['BrierScore'].idxmin(), 'Fold']})")
print(f"  Max Brier: {cv_df['BrierScore'].max():.4f} (Fold {cv_df.loc[cv_df['BrierScore'].idxmax(), 'Fold']})")


Overall OOF Brier Score: 0.1427
Stage 1 (2022-2025) Brier Score: 0.1382

CV Results Summary:
  Mean Brier: 0.1427
  Std Brier: 0.0175
  Min Brier: 0.1134 (Fold 2025)
  Max Brier: 0.1745 (Fold 1998)


## 4. Train Final Model

Train on all data for generating Stage 1 and Stage 2 predictions. Use the median best round from CV as the number of boosting rounds.

In [10]:
final_rounds = int(cv_df['BestRound'].median())
print(f"Final model rounds: {final_rounds} (median of CV best rounds)")

X_all = train[feature_cols]
y_all = train['Label']

dtrain_all = xgb.DMatrix(X_all, label=y_all)
final_model = xgb.train(xgb_params, dtrain_all, num_boost_round=final_rounds)

Final model rounds: 105 (median of CV best rounds)


## 5. Calibration with Isotonic Regression

XGBoost probabilities often cluster around 0.5. We fit isotonic regression on OOF predictions to improve calibration, then apply it to final predictions.

In [11]:
# Fit calibrator on OOF predictions
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_df['Pred'].values, oof_df['Label'].values)

# Check calibration improvement on OOF
oof_df['PredCalibrated'] = calibrator.predict(oof_df['Pred'].values)
calibrated_brier = brier_score_loss(oof_df['Label'], oof_df['PredCalibrated'])

print(f"OOF Brier (raw): {overall_brier:.4f}")
print(f"OOF Brier (calibrated): {calibrated_brier:.4f}")
print(f"Improvement: {overall_brier - calibrated_brier:+.4f}")

OOF Brier (raw): 0.1427
OOF Brier (calibrated): 0.1395
Improvement: +0.0032


## 6. Generate Predictions

In [12]:
# Stage 1 predictions
dstage1 = xgb.DMatrix(stage1[feature_cols])
stage1['Pred_xgboost'] = final_model.predict(dstage1)
stage1['Pred_xgboost_calibrated'] = calibrator.predict(stage1['Pred_xgboost'].values)

# Clip to avoid extreme probabilities
stage1['Pred_xgboost_calibrated'] = stage1['Pred_xgboost_calibrated'].clip(0.02, 0.98)

print(f"Stage 1 predictions: {stage1.shape}")
print(f"  Pred range: [{stage1['Pred_xgboost_calibrated'].min():.3f}, {stage1['Pred_xgboost_calibrated'].max():.3f}]")
print(f"  Pred mean: {stage1['Pred_xgboost_calibrated'].mean():.3f}")

# Evaluate on actual Stage 1 games
stage1_actual = stage1.dropna(subset=['Label'])
if len(stage1_actual) > 0:
    s1_brier_raw = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_xgboost'])
    s1_brier_cal = brier_score_loss(stage1_actual['Label'], stage1_actual['Pred_xgboost_calibrated'])
    print(f"\n  Stage 1 Brier (raw): {s1_brier_raw:.4f}")
    print(f"  Stage 1 Brier (calibrated): {s1_brier_cal:.4f}")

Stage 1 predictions: (258131, 64)
  Pred range: [0.020, 0.980]
  Pred mean: 0.107

  Stage 1 Brier (raw): 0.1258
  Stage 1 Brier (calibrated): 0.1248


In [13]:
# Stage 2 predictions
dstage2 = xgb.DMatrix(stage2[feature_cols])
stage2['Pred_xgboost'] = final_model.predict(dstage2)
stage2['Pred_xgboost_calibrated'] = calibrator.predict(stage2['Pred_xgboost'].values)
stage2['Pred_xgboost_calibrated'] = stage2['Pred_xgboost_calibrated'].clip(0.02, 0.98)

print(f"Stage 2 predictions: {stage2.shape}")
print(f"  Pred range: [{stage2['Pred_xgboost_calibrated'].min():.3f}, {stage2['Pred_xgboost_calibrated'].max():.3f}]")
print(f"  Pred mean: {stage2['Pred_xgboost_calibrated'].mean():.3f}")

Stage 2 predictions: (65703, 63)
  Pred range: [0.020, 0.387]
  Pred mean: 0.083


## 7. Feature Importance

In [14]:
importance = final_model.get_score(importance_type='gain')
imp_df = pd.DataFrame({'Feature': importance.keys(), 'Gain': importance.values()})
imp_df = imp_df.sort_values('Gain', ascending=False)

print("Feature Importance (gain):")
for _, row in imp_df.iterrows():
    print(f"  {row['Feature']:30s}: {row['Gain']:.4f}")

Feature Importance (gain):
  SeedDiff                      : 58.7834
  SeedB                         : 34.5021
  SeedA                         : 31.4037
  AvgPointDiffDiff              : 22.0558
  IsPowerConfDiff               : 12.6553
  NetEffDiff                    : 8.0750
  WinPctDiff                    : 6.4060
  FGPctDiff                     : 5.8090
  OffEffDiff                    : 5.7812
  AvgBlkDiff                    : 5.5259
  FG3PctDiff                    : 5.1016
  DefEffDiff                    : 4.9427
  AvgORDiff                     : 4.5844
  AvgAstDiff                    : 4.5418
  OppFGPctDiff                  : 4.3783
  AvgPossDiff                   : 3.9085
  AvgStlDiff                    : 3.8478
  AvgDRDiff                     : 3.1273
  FTPctDiff                     : 2.9561
  AvgTODiff                     : 2.7044
  OppFG3PctDiff                 : 2.2161


## 8. Save Outputs

In [15]:
outputs = {
    'oof_predictions': oof_df[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Fold', 'IsStage1Val', 'Pred', 'PredCalibrated']],
    'stage1_predictions': stage1[['ID', 'Season', 'TeamA', 'TeamB', 'Label', 'Pred_xgboost', 'Pred_xgboost_calibrated']],
    'stage2_predictions': stage2[['ID', 'Season', 'TeamA', 'TeamB', 'Pred_xgboost', 'Pred_xgboost_calibrated']],
    'cv_results': cv_df,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/womens/oof_predictions.parquet ((1717, 9))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/womens/stage1_predictions.parquet ((258131, 7))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/womens/stage2_predictions.parquet ((65703, 6))
Saved to S3: s3://march-machine-learning-mania-2026/05_models/xgboost/womens/cv_results.parquet ((27, 4))


## 9. Summary

In [16]:
print("=" * 60)
print("XGBOOST MODEL SUMMARY — WOMEN'S")
print("=" * 60)
print(f"\nOOF Brier Score (raw): {overall_brier:.4f}")
print(f"OOF Brier Score (calibrated): {calibrated_brier:.4f}")
if stage1_brier:
    print(f"Stage 1 Brier Score: {stage1_brier:.4f}")
print(f"\nCV Folds: {len(cv_df)}")
print(f"Mean Brier: {cv_df['BrierScore'].mean():.4f} +/- {cv_df['BrierScore'].std():.4f}")
print(f"Final model rounds: {final_rounds}")
print(f"\nTop 5 features (gain):")
for _, row in imp_df.head().iterrows():
    print(f"  {row['Feature']}: {row['Gain']:.4f}")

XGBOOST MODEL SUMMARY — WOMEN'S

OOF Brier Score (raw): 0.1427
OOF Brier Score (calibrated): 0.1395
Stage 1 Brier Score: 0.1382

CV Folds: 27
Mean Brier: 0.1427 +/- 0.0175
Final model rounds: 105

Top 5 features (gain):
  SeedDiff: 58.7834
  SeedB: 34.5021
  SeedA: 31.4037
  AvgPointDiffDiff: 22.0558
  IsPowerConfDiff: 12.6553
